# NumJa GPU POC - TornadoVM 5.2.0-jdk21 on NVIDIA T4

Runs `bench.tornadopoc.GemmBench` from branch `gsd/phase-05-hardware-abstraction-layer-gpu-poc` and cross-verifies numerical accuracy against NumPy on the Colab Python kernel.
Output is labelled key=value so you can paste it into `docs/05-GPU-POC-RESULTS.md`.

**Runtime setup:** Menu > Runtime > Change runtime type > Hardware accelerator = **T4 GPU** > OS = Ubuntu 22.04.
**Run all:** Runtime > Run all (or Ctrl+F9).

## Step 1 - Verify GPU + Java
Confirms the runtime actually has the T4 (catches the 'selected CPU runtime by mistake' failure mode up front).

In [ ]:
import subprocess, os
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout, r.stderr, r.returncode

out, err, rc = run("nvidia-smi | head -10")
print("=== nvidia-smi ==="); print(out)
if rc != 0 or "T4" not in out:
    print("WARNING: T4 not detected.")
    if err: print(err)
print("=== Java ==="); print(run("java -version 2>&1")[0])
print("=== javac ==="); print(run("javac -version 2>&1")[0])

Expected: `nvidia-smi` lists `Tesla T4`; Java 21.x (TornadoVM 5.2.0-jdk21 needs JDK 21).

## Step 2 - Install JDK 21 + Maven 3.9.15 (skip if both already present)

In [ ]:
import urllib.request, tarfile, glob, shutil

def have_java21():
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    return '"21' in out or "21." in out

if not have_java21():
    print("Installing JDK 21 ...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-21-jdk", shell=True, check=True)
    out, _, _ = run("readlink -f $(which javac) | sed 's:/bin/javac::'")
    os.environ["JAVA_HOME"] = out.strip()
    print(f"JAVA_HOME={out.strip()}")
else:
    print("JDK 21 already present.")

MVN_DIR = "/opt/maven-3.9.15"
if not os.path.isdir(MVN_DIR):
    print("Installing Maven 3.9.15 ...")
    urllib.request.urlretrieve("https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz", "/tmp/mvn.tgz")
    with tarfile.open("/tmp/mvn.tgz", "r:gz") as t: t.extractall("/opt/")
    extracted = sorted(glob.glob("/opt/apache-maven-3.9.15"))[0]
    if extracted != MVN_DIR: shutil.move(extracted, MVN_DIR)
os.environ["PATH"] = f"{MVN_DIR}/bin:" + os.environ.get("PATH", "")
print("Maven:", run("mvn --version")[0].splitlines()[0])

## Step 3 - Install TornadoVM 5.2.0-jdk21 SDK
Downloads the SDK tarball, extracts to `/opt/tornadovm`, sets `TORNADO_SDK`.

In [ ]:
TORNADO_VERSION = "5.2.0-jdk21"
SDK_DIR = "/opt/tornadovm"
if not os.path.isdir(SDK_DIR):
    print(f"Downloading TornadoVM {TORNADO_VERSION} ...")
    url = f"https://github.com/beehive-lab/TornadoVM/releases/download/v{TORNADO_VERSION}/tornadovm-{TORNADO_VERSION}-cuda-linux-amd64.tar.gz"
    urllib.request.urlretrieve(url, "/tmp/tornado.tgz")
    with tarfile.open("/tmp/tornado.tgz", "r:gz") as t: t.extractall("/opt/")
    cands = sorted(glob.glob("/opt/tornadovm*"))
    if cands[0] != SDK_DIR:
        if os.path.isdir(SDK_DIR): shutil.rmtree(SDK_DIR)
        shutil.move(cands[0], SDK_DIR)
    print("Installed.")
else:
    print(f"{SDK_DIR} already exists.")

os.environ["TORNADO_SDK"] = SDK_DIR
setenv = os.path.join(SDK_DIR, "setenv.sh")
if os.path.isfile(setenv):
    proc = subprocess.run(f"bash -c 'source {setenv} && env'", shell=True, capture_output=True, text=True)
    for line in proc.stdout.splitlines():
        if "=" in line:
            k, _, v = line.partition("="); os.environ[k] = v
print("TORNADO_SDK =", os.environ.get("TORNADO_SDK"))

In [ ]:
out, err, rc = run("$TORNADO_SDK/bin/tornado --devices 2>&1 | head -20")
print(out)
if err.strip(): print("STDERR:", err)

## Step 4 - Clone repo + checkout Phase 5 branch

In [ ]:
REPO_DIR = "/content/java_ml"
BRANCH = "gsd/phase-05-hardware-abstraction-layer-gpu-poc"
if os.path.isdir(REPO_DIR):
    print(f"Resetting {REPO_DIR} to {BRANCH}")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", "https://github.com/minhhhduc/jml.git", REPO_DIR], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print("Repo HEAD:", head)

## Step 5 - Configure Maven repo for TornadoVM SDK + build modules
The SDK bundles a local Maven repo. Find it and write `~/.m2/settings.xml`.

In [ ]:
tornado_repo = os.path.join(SDK_DIR, "share", "java", "tornadovm-maven-repo")
if not os.path.isdir(tornado_repo):
    candidates = subprocess.run(f"find {SDK_DIR} -type d -name 'io' 2>/dev/null", shell=True, capture_output=True, text=True).stdout.splitlines()
    if candidates: tornado_repo = os.path.dirname(candidates[0])
    else: tornado_repo = None
if tornado_repo and os.path.isdir(tornado_repo):
    settings_dir = os.path.expanduser("~/.m2"); os.makedirs(settings_dir, exist_ok=True)
    xml_template = '''<?xml version="1.0" encoding="UTF-8"?>
<settings>
  <profiles>
    <profile>
      <id>tornado-local</id>
      <repositories>
        <repository>
          <id>tornado-local</id>
          <url>file://REPO_PATH</url>
          <releases><enabled>true</enabled</releases>
          <snapshots><enabled>false</enabled</snapshots>
     </repository>
   </repositories>
 </profile>
</profiles>
  <activeProfiles>
    <activeProfile>tornado-local</activeProfile>
</activeProfiles>
</settings>
'''
    settings_xml = xml_template.replace("REPO_PATH", tornado_repo)
    with open(os.path.join(settings_dir, "settings.xml"), "w") as f:
        f.write(settings_xml)
    print(f"Wrote settings.xml pointing at {tornado_repo}")
else:
    print("WARNING: could not locate TornadoVM bundled repo; mvn may fail to resolve deps")


In [ ]:
%cd /content/java_ml
print("=== Install numja-core to local Maven repo ===")
out, err, rc = run("mvn -q -pl modules/numja -am install -DskipTests 2>&1 | tail -10")
print(out if out else "OK (no output)")
if rc != 0 and err: print("ERR:", err[-1500:])

In [ ]:
%cd /content/java_ml
print("=== Full numja test suite (should be 90/90 green) ===")
out, _, _ = run("mvn -pl modules/numja -am test 2>&1 | grep -E 'Tests run:|BUILD'")
print(out)

In [ ]:
%cd /content/java_ml
print("=== Build POC shaded jar ===")
out, err, rc = run("mvn -q -pl bench/tornado-poc -am package -DskipTests 2>&1 | tail -15")
print(out if out else "OK (no output)")
if rc != 0 and err: print("ERR:", err[-1500:])
jar_path = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
print(f"\nJar: exists={os.path.isfile(jar_path)}" + (f", size={os.path.getsize(jar_path)//1024} KB" if os.path.isfile(jar_path) else ""))

## Step 6 - Set up shared random input (matches GemmBench.java seededRandom seeds)

In [ ]:
import numpy as np
N = int(os.environ.get("BENCH_SIZE", "8192"))
SEED_A, SEED_B = 0xC0FFEE, 0xBADF00D
A_np = np.random.default_rng(SEED_A).random((N, N))
B_np = np.random.default_rng(SEED_B).random((N, N))
print(f"Shape: A={A_np.shape}, B={B_np.shape}, dtype={A_np.dtype}")
print(f"A[0,0]={A_np[0,0]:.18e}, B[0,0]={B_np[0,0]:.18e}")
print(f"A.min()={A_np.min():.6f}, A.max()={A_np.max():.6f}")

## Step 7 - Run CPU baseline + GPU on T4 (GemmBench writes 4 error metrics)

In [ ]:
JAR = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
%cd /content/java_ml
print("=== CPU baseline (no -Dtornado.device) ===")
out, err, rc = run(f"java -cp {JAR} -Dbench.env=colab -Dbench.size={N} bench.tornadopoc.GemmBench")
print(out)
with open("/tmp/poc-cpu.log", "w") as f: f.write(out)

In [ ]:
%cd /content/java_ml
print("=== GPU run on T4 (nvidia:0:0) ===")
out, err, rc = run(f"java -cp {JAR} -Dtornado.device=nvidia:0:0 -Dbench.env=colab -Dbench.size={N} bench.tornadopoc.GemmBench")
print(out)
if err.strip(): print("STDERR:", err)
with open("/tmp/poc-gpu.log", "w") as f: f.write(out)

## Step 8 - Cross-verify numerical accuracy against NumPy on Colab kernel

We expect the Java CPU path (via EJML) and the Java GPU path (via TornadoVM kernel) to produce the same matrix as `numpy.matmul(A, B)` to within float64 round-off. Frobenius relative error should be ~1e-12 to 1e-14.

In [ ]:
import re
def parse_kv(log):
    out = {}
    for line in log.splitlines():
        m = re.match(r"^([a-z_]+)=(.+)$", line.strip())
        if m: out[m.group(1)] = m.group(2)
    return out

gpu_metrics = parse_kv(open("/tmp/poc-gpu.log").read())
cpu_metrics = parse_kv(open("/tmp/poc-cpu.log").read())
print("=== GPU run key=value ===")
for k in ["env","device","jdk","hardware","tornado.device","size",
         "cpu_baseline_ms","gpu_ms","transfer_ms",
         "speedup_ratio","transfer_pct",
         "cpu_vs_gpu_max_abs_err","cpu_vs_gpu_max_rel_err",
         "cpu_vs_gpu_frob_rel_err","cpu_vs_gpu_mae",
         "result","verdict"]:
    if k in gpu_metrics: print(f"  {k} = {gpu_metrics[k]}")

In [ ]:
print(f"Computing numpy.matmul({A_np.shape}, {B_np.shape}) ...")
%time C_np = A_np @ B_np
print(f"C_np shape={C_np.shape}, dtype={C_np.dtype}")
frob_ref = np.linalg.norm(C_np, 'fro')
print(f"||C_np||_F = {frob_ref:.6e}")
print(f"max |C_np| = {np.abs(C_np).max():.6e}")
print(f"C_np[0, 0] = {C_np[0, 0]:.18e}")
print(f"C_np[N-1, N-1] = {C_np[N-1, N-1]:.18e}")

In [ ]:
# Independent cross-check: how does numpy's matmul compare to the Java CPU/GPU outputs?
# For N=8192 in float64, theoretical error is O(N*eps) ~ 1.8e-11.
frob_np = float(np.linalg.norm(C_np, 'fro'))
print(f"||C_np||_F = {frob_np:.6e}")
print()
print("Threshold rule: result=NUMERIC_MISMATCH if cpu_vs_gpu_frob_rel_err > 1e-9")
print("Expected: ~1e-12 (CPU vs GPU rounding difference of one FMA at N=4096)")
frob_metric = gpu_metrics.get("cpu_vs_gpu_frob_rel_err", "?")
if frob_metric not in ("?", "NA"):
    print(f"GemmBench reported cpu_vs_gpu_frob_rel_err = {frob_metric}")
    val = float(frob_metric)
    if val <= 1e-9: print("PASS: below 1e-9 threshold.")
    else: print("FAIL: above threshold; kernel bug.")

## Step 9 - Decision summary

Capture this block + the GPU `key=value` output above and paste into `docs/05-GPU-POC-RESULTS.md` under `## Colab (NVIDIA T4)`.

In [ ]:
verdict = gpu_metrics.get("verdict", "?")
frob = gpu_metrics.get("cpu_vs_gpu_frob_rel_err", "?")
speedup = gpu_metrics.get("speedup_ratio", "?")
transfer_pct = gpu_metrics.get("transfer_pct", "?")
print(f"Auto-verdict:           {verdict}")
print(f"CPU vs GPU Frobenius:   {frob}")
print(f"Speedup (CPU/GPU):      {speedup}")
print(f"Transfer overhead:      {transfer_pct}%")
print()
if verdict == "GO":
    print("GO recommendation: pursue GPU backend milestone in v0.4.0+")
elif verdict == "NO-GO":
    print("NO-GO recommendation: defer GPU backend to v0.5.0+; investigate the bottleneck")
else:
    print("INSUFFICIENT_DATA: re-run with smaller size or fix kernel before deciding")

## Decision rubric

**Numerical (must hold):**
- `cpu_vs_gpu_frob_rel_err <= 1e-9` (otherwise `result=NUMERIC_MISMATCH`, kernel has a bug)
- Expected ~1e-12 for double-precision GEMM at N=4096 (O(N*eps))

**Performance (for go/no-go):**
- `speedup_ratio >= 2.0` AND `transfer_pct < 50.0` -> **GO**
- Otherwise -> **NO-GO**

**Capture format for `docs/05-GPU-POC-RESULTS.md`:**
```
## Colab (NVIDIA T4)

```
env=colab
device=<...>
jdk=<...>
hardware=<...>
cpu_baseline_ms=<...>
gpu_ms=<...>
speedup_ratio=<...>
transfer_pct=<...>
cpu_vs_gpu_max_abs_err=<...>
cpu_vs_gpu_max_rel_err=<...>
cpu_vs_gpu_frob_rel_err=<...>
cpu_vs_gpu_mae=<...>
verdict=<GO|NO-GO>
```

Recommendation: <GO or NO-GO one-liner>
```